# Pythia 160m

In [12]:
import sys
sys.path.insert(0, '../../..')
from src.models import load_model, unload

### Load Model

In [13]:
model_name = "pythia-160m"
model, info = load_model(model_name)

Loaded pretrained model pythia-160m into HookedTransformer
Loaded pythia-160m on mps
  12 layers | 12 heads | d_model=768 | d_mlp=3072 | parallel attn+MLP


### Baseline Prompt

In [14]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a device that allows a user to view a screen

Cached 219 different activation points!


### Declarative Prompts

In [15]:
test_prompts = [
    "A screen reader is",
    "WCAG stands for", 
    "A skip link is",
    "The purpose of alt text is",
    "ARIA stands for",
    "A focus indicator is",
    "Keyboard navigation allows",
    "Color contrast is important because",
    "Semantic HTML helps",
    "Captions are used for",
]

for prompt in test_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a device that allows a user to view a screen
WCAG stands for                →  the International Commission on Geographic Names (ICCG
A skip link is                 →  a link to a page that is not a link
The purpose of alt text is     →  to provide a means for the user to communicate with
ARIA stands for                →  the first time in the history of the country,
A focus indicator is           →  a set of indicators that can be used to measure
Keyboard navigation allows     →  you to navigate through the world with ease.

Color contrast is important because →  it is the most important factor in determining the color
Semantic HTML helps            →  you understand the content of your website.


Captions are used for          →  the following purposes:

The following is a


### Evaluative Prompts

In [16]:
code_prompts = [
    "The following code is not accessible because it doesn't have what? <img src='photo.jpg'>",
    "A <div> with onclick is not accessible because",
    "The accessibility problem with <a href='#'></a> is",
    "<input type='text'> needs a",
    "A button that only says 'Click here' is bad because",
]
for prompt in code_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

I'm trying to create a simple image that will be displayed on a page. I'm
A <div> with onclick is not accessible because →  it is not a <div>
<div> with onclick is not accessible because it is not
The accessibility problem with <a href='#'></a> is →  that it is not possible to access the content of the page.

The accessibility problem with <
<input type='text'> needs a    →  password
<input type='password' name='password' value='password' />
<input
A button that only says 'Click here' is bad because →  it's not a button.

A button that only says 'Click here' is bad because


### Perplexity

In [17]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    
    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()
    
    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 106.73261260986328
Wrong: 41.36967086791992


### Attention Binding

In [28]:
import pandas as pd

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

df = pd.DataFrame(rows)
df.to_csv(f"../results/{model_name}/attention-binding.csv", index=False)


print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")    

print(f"\nFound {len(rows)} heads above threshold")
print(f"Saved to {out_path}")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]

Top 10 by binding strength:
Layer 11, Head  8: 1.0
Layer 11, Head  6: 0.9712
Layer  3, Head  0: 0.9243
Layer  3, Head  2: 0.9156
Layer  1, Head  1: 0.7337
Layer  7, Head  7: 0.6743
Layer  0, Head  0: 0.6281
Layer  1, Head  2: 0.6271
Layer  2, Head 11: 0.6195
Layer  2, Head  8: 0.5459

Found 46 heads above threshold
Saved to ../results/model_name/figures/attention-L11.html


In [34]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 11
attention = cache["pattern", layer]  # shape: [heads, seq, seq]
print(f"Layer 11, Head 8")

html = cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

# Save interactive HTML
out_path = "../results/pythia-160m/figures/attention-L11.html"
with open(out_path, "w") as f:
    f.write(str(html))
print(f"Saved to {out_path}")

html

Layer 11, Head 8
Saved to ../results/pythia-160m/figures/attention-L11.html


In [35]:
unload(model)

Model unloaded, memory cleared
